<a href="https://colab.research.google.com/github/atilimai/plant-ai-project/blob/main/notebooks/Boran_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets -q
import os
import pandas as pd
from datasets import load_dataset

raw_dataset = load_dataset("mohanty/PlantVillage", name="default")
df = pd.DataFrame(raw_dataset['train'])

print(f"Dataset loaded from Hugging Face.")
print(f"Total images found: {len(df)}")

In [ ]:
import numpy as np


if 'path' in df.columns:
    import re
    df['leaf_id'] = df['path'].apply(lambda x: re.search(r'leaf\d+', str(x)).group(0) if re.search(r'leaf\d+', str(x)) else str(x))
else:
    df['leaf_id'] = [f"inst_{i}" for i in range(len(df))]

print(f"Leaf/Instance identification complete.")
print(f"Unique groups identified: {df['leaf_id'].nunique()}")

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['leaf_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

intersection = set(train_df['leaf_id']).intersection(set(test_df['leaf_id']))

print(f"Dataset split successfully using GroupShuffleSplit.")
print(f"Train Set: {len(train_df)} | Test Set: {len(test_df)}")
print(f"Leakage Test Result: {len(intersection)} (0 means success)")

In [ ]:
os.makedirs('data/splits', exist_ok=True)

train_df.to_csv('data/splits/train_split.csv', index=False)
test_df.to_csv('data/splits/test_split.csv', index=False)

print(f"Metadata files created in data/splits/")
print(os.listdir('data/splits'))